<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 11: BI AND AI APPLICATIONS</div><div style="color:#17212b;font-size:30px;font-weight:750">Deliver reviewed metrics to BI and AI consumers through stable SQL contracts</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 1 course database. Every result is rendered as a table and every write is scoped to this module's objects.</p></div>

## Boundary

This lab never changes the Level 1 source tables. It creates or replaces only objects with the `_l2` suffix.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_l2")
lab.execute("""
CREATE VIEW bi_order_metrics_l2 AS
SELECT order_date,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date
""")
lab.sql("SELECT * FROM bi_order_metrics_l2 ORDER BY order_date", title="BI semantic view")

In [ ]:
lab.sql("""
SELECT order_date,
       order_count AS feature_order_count,
       gross_amount AS feature_gross_amount,
       average_order_amount AS feature_average_amount,
       CASE WHEN order_count = 0 THEN 1 ELSE 0 END AS zero_order_flag
FROM bi_order_metrics_l2
ORDER BY order_date
""", title="Reviewed feature projection")

In [ ]:
lab.sql("""
SELECT
    COUNT(*) AS serving_days,
    SUM(CASE WHEN order_count IS NULL THEN 1 ELSE 0 END) AS null_order_days,
    MIN(order_date) AS first_date,
    MAX(order_date) AS last_date,
    SUM(order_count) AS served_orders
FROM bi_order_metrics_l2
""", title="Consumer contract checks", final=True)

# A BI dashboard or AI application consumes this result through its own connector.
# Doris provides the reviewed rows; it does not silently define model accuracy.

## Takeaway

Compare the result with the business grain stated in the lesson. A successful SQL statement is not by itself evidence that the model, metric, access boundary, or consumer contract is correct.